# Multilayer Perceptron classifier

Trains and evaluates the MLP neural-network classifier.

**Input data:** place the standardized Excel file at `../data/processed/Pyrite_Standarized_data_file_New_Paper.xlsx`.

**Note:** outputs were cleared for GitHub readability and reproducibility.


In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Set random seed
seed = 42
random.seed(seed)
np.random.seed(seed)

# Load dataset
df = pd.read_excel('../data/processed/Pyrite_Standarized_data_file_New_Paper.xlsx')

# Features and target
X = df[['Co', 'Ni', 'Cu', 'Zn', 'Se', 'Ag', 'Sb', 'Pb', 'Bi', 'As']]
y = df['Deposit type']

# Encode target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Function to train & evaluate MLP model with AUC
def evaluate_mlp_model(X_data, y_data, label):
    # Split data into train (60%), val (20%), test (20%)
    X_train, X_temp, y_train, y_temp = train_test_split(X_data, y_data, test_size=0.4, random_state=seed, stratify=y_data)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=seed, stratify=y_temp)

    # Initialize and train MLP classifier
    mlp_classifier = MLPClassifier(
        hidden_layer_sizes=(150, 100, 50),
        max_iter=300,
        activation='tanh',
        solver='adam',
        alpha=0.0001,
        learning_rate='constant',
        random_state=seed
    )
    mlp_classifier.fit(X_train, y_train)

    # Validation set evaluation
    y_val_pred = mlp_classifier.predict(X_val)
    y_val_proba = mlp_classifier.predict_proba(X_val)
    val_auc = roc_auc_score(y_val, y_val_proba, multi_class='ovr')

    print(f"\n--- {label} | Validation ---")
    print(f"Accuracy: {accuracy_score(y_val, y_val_pred) * 100:.2f}%")
    print(f"AUC (OvR): {val_auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
    print("Classification Report:\n", classification_report(y_val, y_val_pred, target_names=label_encoder.classes_))

    # Test set evaluation
    y_test_pred = mlp_classifier.predict(X_test)
    y_test_proba = mlp_classifier.predict_proba(X_test)
    test_auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr')

    print(f"\n--- {label} | Test ---")
    print(f"Accuracy: {accuracy_score(y_test, y_test_pred) * 100:.2f}%")
    print(f"AUC (OvR): {test_auc:.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
    print("Classification Report:\n", classification_report(y_test, y_test_pred, target_names=label_encoder.classes_))

# 1. Original data
evaluate_mlp_model(X_imputed, y_encoded, "Original (No Resampling)")

# 2. SMOTE resampling
smote = SMOTE(random_state=seed)
X_smote, y_smote = smote.fit_resample(X_imputed, y_encoded)
evaluate_mlp_model(X_smote, y_smote, "SMOTE Resampling")

# 3. RUS resampling
rus = RandomUnderSampler(random_state=seed)
X_rus, y_rus = rus.fit_resample(X_imputed, y_encoded)
evaluate_mlp_model(X_rus, y_rus, "Random UnderSampling (RUS)")
